# 🎤 Speaker Verification with ECAPA-TDNN on Kaggle (2x T4 GPUs)

This notebook implements the **ECAPA-TDNN** (Emphasized Channel Attention, Propagation and Aggregation) model for Speaker Verification.

### Pipeline Overview:
1. **Environment Preparation**: Setup dependencies and configure dual GPU training.
2. **Data Pipeline & Augmentation**: Same data pipeline as Notebook 1 for fair comparison.
3. **Model Architecture**: SE-Res2Net blocks, multi-scale feature aggregation, Attentive Statistics Pooling (192-D embeddings).
4. **Training & Validation**: Optimize cosine similarity threshold to maximize **F1-score**.
5. **Visualization**: Loss curves, similarity distributions, ROC curves, and EER.
6. **Export**: Export model for comparison in Notebook 3.

## 🛠️ Step 1: Prepare Environment and Import Libraries

In [ ]:
!pip install -q torchaudio soundfile librosa matplotlib seaborn scikit-learn pandas numpy tqdm

import os, re, sys, time, math, random, glob, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_curve, auc
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f'[INFO] Device: {device} | GPUs: {num_gpus}')
for i in range(num_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
os.makedirs('checkpoints/ecapa', exist_ok=True)
os.makedirs('results', exist_ok=True)

## 🔄 Step 2: Auto-Discover Datasets, Preprocess, and Setup Augmentation
Identical data pipeline to Notebook 1 (same seeds, same splits) for a fair comparison.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

SR = 16000
DURATION = 3
NUM_SAMPLES = SR * DURATION
N_MELS = 80
BATCH_SIZE = 64
NUM_WORKERS = 4

# ============================================================
# AUTO-DISCOVER DATASET PATHS
# ============================================================
INPUT_ROOT = '/kaggle/input'

print('=' * 60)
print('STEP 1: Discovering /kaggle/input/ structure')
print('=' * 60)
for ds in sorted(os.listdir(INPUT_ROOT)):
    ds_path = os.path.join(INPUT_ROOT, ds)
    if os.path.isdir(ds_path):
        print(f'\n  [DIR] {ds}/')
        for sub in sorted(os.listdir(ds_path))[:15]:
            sub_path = os.path.join(ds_path, sub)
            tag = '[DIR]' if os.path.isdir(sub_path) else '[FILE]'
            print(f'    {tag} {sub}')
            if os.path.isdir(sub_path):
                for sub2 in sorted(os.listdir(sub_path))[:8]:
                    sub2_path = os.path.join(sub_path, sub2)
                    tag2 = '[DIR]' if os.path.isdir(sub2_path) else '[FILE]'
                    print(f'      {tag2} {sub2}')

print(f'\n{"=" * 60}')
print('STEP 2: Searching for all audio files...')
print('=' * 60)
all_wav = glob.glob(os.path.join(INPUT_ROOT, '**', '*.wav'), recursive=True)
all_flac = glob.glob(os.path.join(INPUT_ROOT, '**', '*.flac'), recursive=True)
print(f'  .wav  files found: {len(all_wav)}')
print(f'  .flac files found: {len(all_flac)}')
all_audio = all_wav + all_flac
print(f'  Total audio files: {len(all_audio)}')

if len(all_audio) > 0:
    print('\n  Sample paths:')
    for p in all_audio[:10]:
        print(f'    {p}')

# Classify: VoxCeleb vs MUSAN
print(f'\n{"=" * 60}')
print('STEP 3: Classifying audio files')
print('=' * 60)
vox_wav_files = []
noise_files = []
for f in all_audio:
    fp = f.replace('\\', '/')
    if re.search(r'/id\d{3,}/', fp):
        vox_wav_files.append(f)
    elif 'noise' in fp.lower() and 'musan' in fp.lower():
        noise_files.append(f)

if len(vox_wav_files) == 0:
    print('  [WARN] No VoxCeleb files by id pattern. Trying broader match...')
    for f in all_audio:
        fp = f.replace('\\', '/').lower()
        if 'vox' in fp or 'celeb' in fp:
            vox_wav_files.append(f)
if len(noise_files) == 0:
    print('  [WARN] No MUSAN noise files. Trying broader match...')
    for f in all_audio:
        fp = f.replace('\\', '/').lower()
        if 'noise' in fp:
            noise_files.append(f)

print(f'  VoxCeleb files: {len(vox_wav_files)}')
print(f'  MUSAN noise:    {len(noise_files)}')
assert len(vox_wav_files) > 0, 'ERROR: No VoxCeleb files found!'

# Build speaker dictionary
def get_speaker_id(path):
    for p in path.replace('\\', '/').split('/'):
        if re.match(r'^id\d{3,}$', p): return p
    return 'unknown'

speaker_dict = {}
for path in vox_wav_files:
    spk = get_speaker_id(path)
    if spk != 'unknown':
        speaker_dict.setdefault(spk, []).append(path)

if len(speaker_dict) == 0:
    print('  [WARN] No idXXXXX folders. Using parent dirs as speaker labels.')
    for path in vox_wav_files:
        parts = path.replace('\\', '/').split('/')
        if len(parts) >= 3:
            speaker_dict.setdefault(parts[-3], []).append(path)

print(f'  Unique speakers: {len(speaker_dict)}')

SELECT_SUBSET_SPEAKERS = None
if SELECT_SUBSET_SPEAKERS is not None and len(speaker_dict) > SELECT_SUBSET_SPEAKERS:
    sorted_spk = sorted(speaker_dict.items(), key=lambda x: len(x[1]), reverse=True)
    speaker_dict = dict(sorted_spk[:SELECT_SUBSET_SPEAKERS])
    vox_wav_files = [f for files in speaker_dict.values() for f in files]
    print(f'  Selected top {SELECT_SUBSET_SPEAKERS} speakers ({len(vox_wav_files)} files)')

speakers = sorted(speaker_dict.keys())
spk_to_label = {s: i for i, s in enumerate(speakers)}
label_to_spk = {i: s for i, s in enumerate(speakers)}
NUM_CLASSES = len(speakers)
print(f'  NUM_CLASSES = {NUM_CLASSES}')

# Noise augmentation
class AudioAugmentation:
    def __init__(self, nf, sr=16000):
        self.nf = nf; self.sr = sr
    def add_noise(self, w, snr=None):
        if not self.nf: return w
        if snr is None: snr = random.uniform(0, 15)
        try: nw, _ = librosa.load(random.choice(self.nf), sr=self.sr)
        except: return w
        if len(nw)<len(w): nw = np.tile(nw, math.ceil(len(w)/len(nw)))[:len(w)]
        else: s=random.randint(0,len(nw)-len(w)); nw=nw[s:s+len(w)]
        sc = math.sqrt((np.mean(w**2)+1e-8)/(np.mean(nw**2)+1e-8)*10**(-snr/10))
        mx = w + sc*nw
        m = np.max(np.abs(mx))
        return mx/m if m>1 else mx

augmenter = AudioAugmentation(noise_files, SR)

# Train/Val split
train_files, val_files = [], []
for spk, files in speaker_dict.items():
    if len(files) >= 4:
        tr, va = train_test_split(files, test_size=0.25, random_state=42)
        train_files.extend([(f, spk) for f in tr])
        val_files.extend([(f, spk) for f in va])
    else:
        train_files.extend([(f, spk) for f in files])
        val_files.extend([(f, spk) for f in files])
print(f'  Train: {len(train_files)} | Val: {len(val_files)}')

# Dataset
class VoxCelebDataset(Dataset):
    def __init__(self, pairs, sr=16000, dur=3, augment=False, augmenter=None):
        self.pairs=pairs; self.sr=sr; self.ns=sr*dur; self.aug=augment; self.augm=augmenter
        self.mel = T.MelSpectrogram(sample_rate=sr, n_fft=400, win_length=400, hop_length=160, n_mels=80)
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        fp, spk = self.pairs[i]
        label = spk_to_label[spk]
        try: w, _ = librosa.load(fp, sr=self.sr)
        except: w = np.zeros(self.ns, dtype=np.float32)
        if len(w)<self.ns: w=np.pad(w,(0,self.ns-len(w)))
        else:
            s = random.randint(0,len(w)-self.ns) if self.aug else (len(w)-self.ns)//2
            w = w[s:s+self.ns]
        if self.aug and self.augm and random.random()<0.6: w=self.augm.add_noise(w)
        wt=torch.tensor(w,dtype=torch.float32)
        lm=torch.log(self.mel(wt)+1e-6); lm=lm-lm.mean()
        return lm, label

train_dataset = VoxCelebDataset(train_files, SR, DURATION, True, augmenter)
val_dataset = VoxCelebDataset(val_files, SR, DURATION, False)
train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'  DataLoaders ready. Train: {len(train_loader)} | Val: {len(val_loader)}')

# Verification pairs
def gen_pairs(fl, n=1000, seed=42):
    random.seed(seed); pairs=[]; d={}
    for f,s in fl: d.setdefault(s,[]).append(f)
    spks=list(d.keys()); att=0
    while len(pairs)<n//2 and att<10000:
        att+=1; s=random.choice(spks)
        if len(d[s])>=2: a,b=random.sample(d[s],2); pairs.append((a,b,1))
    att=0
    while len(pairs)<n and att<10000:
        att+=1; s1,s2=random.sample(spks,2)
        pairs.append((random.choice(d[s1]),random.choice(d[s2]),0))
    return pairs

val_pairs = gen_pairs(val_files, 1000, 42)
test_pairs = gen_pairs(val_files, 1000, 999)
print(f'  Val pairs: {len(val_pairs)} | Test pairs: {len(test_pairs)}')

## 🧠 Step 3: ECAPA-TDNN Architecture
SE-Res2Net blocks, multi-scale feature aggregation, Attentive Statistics Pooling, 192-D embeddings.

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, ch//r), nn.ReLU(), nn.Linear(ch//r, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=2)).unsqueeze(2)

class SERes2NetBlock(nn.Module):
    def __init__(self, ch, ks, dil, scale=4):
        super().__init__()
        self.scale = scale
        self.conv1 = nn.Conv1d(ch, ch, 1); self.bn1 = nn.BatchNorm1d(ch)
        w = ch // scale
        self.convs = nn.ModuleList([nn.Conv1d(w, w, ks, dilation=dil, padding=int(dil*(ks-1)/2)) for _ in range(scale-1)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(w) for _ in range(scale-1)])
        self.conv3 = nn.Conv1d(ch, ch, 1); self.bn3 = nn.BatchNorm1d(ch)
        self.se = SEBlock(ch)
    def forward(self, x):
        res = x
        out = F.relu(self.bn1(self.conv1(x)))
        sp = torch.chunk(out, self.scale, dim=1)
        ns = [sp[0]]
        for i in range(1, self.scale):
            s = sp[i] + ns[i-1] if i > 1 else sp[i]
            ns.append(F.relu(self.bns[i-1](self.convs[i-1](s))))
        out = self.se(self.bn3(self.conv3(torch.cat(ns, dim=1))))
        return F.relu(out + res)

class AttentiveStatsPooling(nn.Module):
    def __init__(self, inc, att_ch=128):
        super().__init__()
        self.c1 = nn.Conv1d(inc, att_ch, 1)
        self.c2 = nn.Conv1d(att_ch, inc, 1)
    def forward(self, x):
        a = self.c2(torch.tanh(self.c1(x)))
        w = F.softmax(a, dim=2)
        mu = (x * w).sum(dim=2)
        sg = torch.sqrt(torch.clamp((w * (x - mu.unsqueeze(2))**2).sum(dim=2), min=1e-9))
        return torch.cat([mu, sg], dim=1)

class ECAPATDNNModel(nn.Module):
    def __init__(self, input_dim=80, num_classes=10, embedding_dim=192):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 512, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(512)
        self.l1 = SERes2NetBlock(512, 3, 2)
        self.l2 = SERes2NetBlock(512, 3, 3)
        self.l3 = SERes2NetBlock(512, 3, 4)
        self.conv2 = nn.Conv1d(2048, 1536, 1)
        self.bn2 = nn.BatchNorm1d(1536)
        self.pool = AttentiveStatsPooling(1536)
        self.fc = nn.Linear(3072, embedding_dim)
        self.bnfc = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)
    def extract_embedding(self, x):
        x0 = F.relu(self.bn1(self.conv1(x)))
        x1=self.l1(x0); x2=self.l2(x1); x3=self.l3(x2)
        xc = torch.cat([x0,x1,x2,x3], dim=1)
        xm = F.relu(self.bn2(self.conv2(xc)))
        return self.bnfc(self.fc(self.pool(xm)))
    def forward(self, x):
        return self.classifier(F.relu(self.extract_embedding(x)))

model = ECAPATDNNModel(N_MELS, NUM_CLASSES, 192)
if num_gpus > 1: model = nn.DataParallel(model)
model = model.to(device)
print(f'ECAPA-TDNN built. Classes: {NUM_CLASSES}')

## 📉 Step 4: Training & Verification (F1-Score Optimization)
Includes VRAM cleanup after every epoch.

In [ ]:
class AAMSoftmax(nn.Module):
    def __init__(self, input_dim, num_classes, margin=0.2, scale=30.0):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, input_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, x, label):
        x_norm = F.normalize(x, p=2, dim=1)
        w_norm = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(x_norm, w_norm)
        sine = torch.sqrt(torch.clamp(1.0 - torch.square(cosine), min=1e-9))
        cos_mt = cosine * self.cos_m - sine * self.sin_m
        cos_mt = torch.where(cosine > self.th, cos_mt, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1.0)
        output = (one_hot * cos_mt) + ((1.0 - one_hot) * cosine)
        output = output * self.scale
        return output

EPOCHS = 30
criterion = AAMSoftmax(input_dim=192, num_classes=NUM_CLASSES, margin=0.2, scale=30.0).to(device)
optimizer = optim.AdamW(list(model.parameters()) + list(criterion.parameters()), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=400, win_length=400, hop_length=160, n_mels=80).to(device)

def get_emb(mdl, fp, mt, dev):
    mdl.eval()
    try: w,_=librosa.load(fp,sr=16000)
    except: w=np.zeros(48000,dtype=np.float32)
    ns=48000
    if len(w)<ns: w=np.pad(w,(0,ns-len(w)))
    else: w=w[(len(w)-ns)//2:(len(w)-ns)//2+ns]
    wt=torch.tensor(w,dtype=torch.float32).unsqueeze(0).to(dev)
    with torch.no_grad():
        m=mt(wt); lm=torch.log(m+1e-6); lm=lm-lm.mean()
        e = mdl.module.extract_embedding(lm) if isinstance(mdl,nn.DataParallel) else mdl.extract_embedding(lm)
    return F.normalize(e,p=2,dim=1).cpu().numpy()[0]

def eval_verif(mdl, pairs, mt, dev):
    mdl.eval(); sims=[]; labs=[]; cache={}
    for a,b,lb in pairs:
        if a not in cache: cache[a]=get_emb(mdl,a,mt,dev)
        if b not in cache: cache[b]=get_emb(mdl,b,mt,dev)
        sims.append(np.dot(cache[a],cache[b])); labs.append(lb)
    del cache; gc.collect(); torch.cuda.empty_cache()
    sims,labs = np.array(sims), np.array(labs)
    bt,bf = 0.0,0.0
    for th in np.linspace(-1,1,200):
        f=f1_score(labs,(sims>=th).astype(int))
        if f>bf: bf,bt=f,th
    preds=(sims>=bt).astype(int)
    fpr,tpr,_=roc_curve(labs,sims); fnr=1-tpr
    eer=fpr[np.nanargmin(np.abs(fpr-fnr))]
    return {'threshold':bt,'f1':bf,'precision':precision_score(labs,preds,zero_division=0),
            'recall':recall_score(labs,preds,zero_division=0),'accuracy':accuracy_score(labs,preds),
            'eer':eer,'sims':sims,'labels':labs}

history = {'train_loss':[],'val_loss':[],'val_f1':[],'val_eer':[]}
best_val_f1 = 0.0

for epoch in range(1, EPOCHS+1):
    model.train(); rl=0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for bf, bl in pbar:
        bf,bl = bf.to(device),bl.to(device)
        optimizer.zero_grad()
        # Extract direct 192-D speaker embedding
        emb = model.module.extract_embedding(bf) if isinstance(model, nn.DataParallel) else model.extract_embedding(bf)
        # Get AAM-Softmax cosine logits
        logits = criterion(emb, bl)
        loss = nn.CrossEntropyLoss()(logits, bl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        rl += loss.item()
        pbar.set_postfix(Loss=f'{loss.item():.4f}')
    etl = rl/len(train_loader)
    
    model.eval(); vl=0.0
    with torch.no_grad():
        for bf,bl in val_loader:
            bf,bl=bf.to(device),bl.to(device)
            # Validation classification loss is proxyed using classifier CrossEntropy
            vl += nn.CrossEntropyLoss()(model(bf), bl).item()
    evl = vl/len(val_loader)
    
    vr = eval_verif(model, val_pairs, mel_transform, device)
    history['train_loss'].append(etl); history['val_loss'].append(evl)
    history['val_f1'].append(vr['f1']); history['val_eer'].append(vr['eer'])
    scheduler.step()
    
    print(f'  Loss: {etl:.4f}/{evl:.4f} | F1: {vr["f1"]:.4f} @ {vr["threshold"]:.3f} | EER: {vr["eer"]:.4f}')
    
    if vr['f1'] > best_val_f1:
        best_val_f1 = vr['f1']
        ms = model.module if isinstance(model,nn.DataParallel) else model
        torch.save({'epoch':epoch,'model_state_dict':ms.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(),
                    'val_f1':best_val_f1,'optimal_threshold':vr['threshold'],
                    'spk_to_label':spk_to_label,'label_to_spk':label_to_spk},
                   'checkpoints/ecapa/best_model.pt')
        print(f'  * Best F1! Checkpoint saved.')
    
    gc.collect(); torch.cuda.empty_cache()
    print(f'  [MEM] VRAM cleared.', end=' ')
    if torch.cuda.is_available(): print(f'Alloc: {torch.cuda.memory_allocated()/1e6:.0f}MB')
    else: print()


## 📊 Step 5: Test Set Evaluation & Visualization

In [ ]:
ckpt = 'checkpoints/ecapa/best_model.pt'
if os.path.exists(ckpt):
    c = torch.load(ckpt, map_location=device, weights_only=False)
    eval_model = ECAPATDNNModel(N_MELS, NUM_CLASSES, 192)
    eval_model.load_state_dict(c['model_state_dict'])
    eval_model = eval_model.to(device)
    opt_threshold = c['optimal_threshold']
    print(f'Best model restored. Threshold: {opt_threshold:.3f}')
else:
    eval_model = model; opt_threshold = 0.5

tr = eval_verif(eval_model, test_pairs, mel_transform, device)
ts, tl = tr['sims'], tr['labels']
tp = (ts >= opt_threshold).astype(int)
tf1 = f1_score(tl, tp)
tacc = accuracy_score(tl, tp)
tprec = precision_score(tl, tp)
trec = recall_score(tl, tp)
fpr, tpr, _ = roc_curve(tl, ts)
tauc = auc(fpr, tpr)

print(f'\n{"="*50} TEST REPORT {"="*50}')
print(f'  Threshold: {opt_threshold:.3f}')
print(f'  F1: {tf1:.4f} | Acc: {tacc:.4f} | Prec: {tprec:.4f} | Rec: {trec:.4f}')
print(f'  EER: {tr["eer"]:.4f} | AUC: {tauc:.4f}')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes[0,0].plot(history['train_loss'], label='Train', lw=2)
axes[0,0].plot(history['val_loss'], label='Val', lw=2)
axes[0,0].set_title('Classification Loss (ECAPA)', fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(history['val_f1'], label='F1', color='green', lw=2)
axes[0,1].plot(history['val_eer'], label='EER', color='red', lw=2, ls='--')
axes[0,1].set_title('Verification Metrics (ECAPA)', fontweight='bold'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

sns.histplot(ts[tl==1], color='green', label='Same', kde=True, bins=30, stat='density', alpha=0.4, ax=axes[1,0])
sns.histplot(ts[tl==0], color='red', label='Diff', kde=True, bins=30, stat='density', alpha=0.4, ax=axes[1,0])
axes[1,0].axvline(opt_threshold, color='navy', ls='--', lw=2, label=f'Th={opt_threshold:.3f}')
axes[1,0].set_title('Similarity Distribution (ECAPA)', fontweight='bold'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'AUC={tauc:.4f}')
axes[1,1].plot([0,1],[0,1],'--',color='grey')
axes[1,1].set_title('ROC (ECAPA)', fontweight='bold'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('results/ecapa_evaluation_plots.png', dpi=150); plt.show()

## 💾 Step 6: Export Model for Comparison

## 🔊 Step 5.5: Fixed SNR Noisy Audio Evaluation
As requested, we run standard verification evaluations under fixed noise levels from the MUSAN noise dataset at **0dB, 10dB, and 20dB** to measure the robustness and degradation of our ECAPA-TDNN model compared to the clean baseline.

In [ ]:
def eval_verif_noise(mdl, pairs, mt, dev, snr_val):
    mdl.eval()
    sims, labs = [], []
    cache = {}
    for a, b, lb in tqdm(pairs, desc=f'Evaluating {snr_val}dB'):
        for f in [a, b]:
            if f not in cache:
                try:
                    w, _ = librosa.load(f, sr=16000)
                except Exception:
                    w = np.zeros(48000, dtype=np.float32)
                ns = 48000
                if len(w) < ns: w = np.pad(w, (0, ns-len(w)))
                else: w = w[:ns]
                
                # Mix in random MUSAN background noise at exactly the fixed SNR
                if len(noise_files) > 0:
                    try:
                        nw, _ = librosa.load(random.choice(noise_files), sr=16000)
                        if len(nw) < len(w):
                            nw = np.tile(nw, math.ceil(len(w)/len(nw)))[:len(w)]
                        else:
                            nw = nw[:len(w)]
                        clean_p = np.mean(w**2) + 1e-8
                        noise_p = np.mean(nw**2) + 1e-8
                        scale = math.sqrt(clean_p / noise_p * 10**(-snr_val/10))
                        w = w + scale * nw
                        mx = np.max(np.abs(w))
                        if mx > 1.0: w = w / mx
                    except Exception:
                        pass
                wt = torch.tensor(w, dtype=torch.float32).unsqueeze(0).to(dev)
                with torch.no_grad():
                    m = mt(wt)
                    lm = torch.log(m+1e-6); lm = lm - lm.mean()
                    e = mdl.module.extract_embedding(lm) if isinstance(mdl, nn.DataParallel) else mdl.extract_embedding(lm)
                cache[f] = F.normalize(e, p=2, dim=1).cpu().numpy()[0]
        sims.append(np.dot(cache[a], cache[b]))
        labs.append(lb)
    del cache; gc.collect(); torch.cuda.empty_cache()
    sims, labs = np.array(sims), np.array(labs)
    preds = (sims >= opt_threshold).astype(int)
    fpr, tpr, _ = roc_curve(labs, sims)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    return {'f1': f1_score(labs, preds), 'eer': eer, 'accuracy': accuracy_score(labs, preds),
            'precision': precision_score(labs, preds, zero_division=0), 'recall': recall_score(labs, preds, zero_division=0)}

print('Evaluating ECAPA-TDNN under noise stress test (0dB, 10dB, 20dB)...')
noise_results = {}
for snr in [20, 10, 0]:
    noise_results[snr] = eval_verif_noise(eval_model, test_pairs, mel_transform, device, snr)

print(f'\n{"="*50} NOISE STRESS TEST (ECAPA-TDNN) {"="*50}')
print(f'  Clean Base  -> F1: {tf1:.4f} | EER: {tr["eer"]:.4f} | Accuracy: {tacc:.4f} | Precision: {tprec:.4f} | Recall: {trec:.4f}')
for snr in [20, 10, 0]:
    r = noise_results[snr]
    print(f'  SNR {snr:2d} dB   -> F1: {r["f1"]:.4f} | EER: {r["eer"]:.4f} | Accuracy: {r["accuracy"]:.4f} | Precision: {r["precision"]:.4f} | Recall: {r["recall"]:.4f}')


In [ ]:
torch.save({
    'model_architecture': 'ECAPATDNNModel',
    'input_dim': N_MELS, 'embedding_dim': 192, 'num_classes': NUM_CLASSES,
    'optimal_threshold': opt_threshold,
    'model_state_dict': eval_model.state_dict(),
    'spk_to_label': spk_to_label, 'label_to_spk': label_to_spk,
    'test_metrics': {'f1_score':tf1, 'eer':tr['eer'], 'accuracy':tacc,
                     'precision':tprec, 'recall':trec, 'auc':tauc}
}, 'results/ecapa_tdnn_final_model.pt')
print('[SUCCESS] ECAPA-TDNN exported to results/ecapa_tdnn_final_model.pt')
v = torch.load('results/ecapa_tdnn_final_model.pt', map_location='cpu', weights_only=False)
print(f'  Arch: {v["model_architecture"]} | F1: {v["test_metrics"]["f1_score"]:.4f}')